# Document Clustering for Topic Modeling
**Comparing TF-IDF vs. Sentence-BERT representations, clustered with K-Means and HDBSCAN**

This notebook implements the five-layer architecture from the project proposal:
1. Data Ingestion
2. Preprocessing
3. Feature Extraction (TF-IDF + SBERT, PCA/UMAP)
4. Clustering (K-Means + HDBSCAN, hyperparameter tuning)
5. Evaluation & Visualization (Silhouette, Davies-Bouldin, Calinski-Harabasz, ARI, NMI, interactive scatter plot)

> Run cells top to bottom. The first cell installs everything needed (safe to skip if already installed).

## 0. Setup — install & import dependencies

In [ ]:
# Run once. Safe to re-run.
!pip install -q scikit-learn sentence-transformers umap-learn hdbscan plotly nltk pandas numpy seaborn matplotlib


In [ ]:
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)

import umap
import hdbscan
from sentence_transformers import SentenceTransformer

import plotly.express as px
import plotly.graph_objects as go

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("All libraries loaded.")

## 1. Data Ingestion Layer
Loads the 20 Newsgroups dataset. Ground-truth category labels are kept aside and used **only**
for external evaluation (ARI, NMI) — never shown to the clustering algorithms.

To use a different corpus (e.g. BBC News CSV or your own documents), replace this cell's
body with a loader that returns `raw_df` with the same columns: `doc_id`, `text`, `true_label`.

In [ ]:
# ---- Option A: 20 Newsgroups (default) ----
CATEGORIES = [
    'rec.sport.hockey', 'sci.space', 'talk.politics.mideast',
    'comp.graphics', 'rec.autos', 'sci.med'
]  # subset of 6 categories keeps runtime reasonable; use None for all 20

newsgroups = fetch_20newsgroups(
    subset='all',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=RANDOM_STATE
)

raw_df = pd.DataFrame({
    'doc_id': range(len(newsgroups.data)),
    'text': newsgroups.data,
    'true_label': newsgroups.target
})
label_names = newsgroups.target_names

# Drop empty / very short documents
raw_df['text'] = raw_df['text'].str.strip()
raw_df = raw_df[raw_df['text'].str.len() > 50].reset_index(drop=True)
raw_df = raw_df.drop_duplicates(subset='text').reset_index(drop=True)

print(f"Loaded {len(raw_df)} documents across {len(label_names)} categories: {label_names}")
raw_df.head(3)

In [ ]:
# ---- Option B: your own CSV (uncomment to use instead of 20 Newsgroups) ----
# raw_df = pd.read_csv('your_documents.csv')   # must contain a 'text' column
# raw_df['doc_id'] = range(len(raw_df))
# raw_df['true_label'] = -1   # no ground truth available -> internal metrics only
# label_names = None

## 2. Preprocessing Layer
Cleaning → tokenization → stop-word removal → lemmatization.
A few dataset-specific stop words (e.g. "subject", "writes", "article") are added because
20 Newsgroups posts carry email-style boilerplate even after header removal.

In [ ]:
stop_words = set(stopwords.words('english'))
stop_words.update(['subject', 'writes', 'article', 'would', 'could', 'one', 'also', 'get', 'use'])
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)          # URLs
    text = re.sub(r'\S+@\S+', ' ', text)                    # emails
    text = re.sub(r'[^a-z\s]', ' ', text)                    # punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

raw_df['clean_text'] = raw_df['text'].apply(clean_text)
raw_df = raw_df[raw_df['clean_text'].str.split().str.len() >= 5].reset_index(drop=True)

clean_df = raw_df[['doc_id', 'clean_text', 'true_label']].copy()
print(f"{len(clean_df)} documents remain after cleaning.")
print("\nExample:")
print("RAW  :", raw_df['text'].iloc[0][:150], "...")
print("CLEAN:", clean_df['clean_text'].iloc[0][:150], "...")

## 3. Feature Extraction Layer
Two independent representations are built from the same documents:

- **TF-IDF** — traditional lexical representation (unigrams + bigrams)
- **SBERT** — contextual semantic representation (`all-MiniLM-L6-v2`)

Each is then reduced with **TruncatedSVD/PCA** (for clustering) and **UMAP** (2D, for visualization only).
Clustering always happens on the reduced high-dimensional space, never on the 2D projection.

In [ ]:
# --- TF-IDF ---
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.6,
    max_features=5000,
    sublinear_tf=True
)
X_tfidf = tfidf_vectorizer.fit_transform(clean_df['clean_text'])
print("TF-IDF matrix shape:", X_tfidf.shape)

# Reduce TF-IDF with TruncatedSVD (works directly on sparse matrices)
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_tfidf_reduced = svd.fit_transform(X_tfidf)
X_tfidf_reduced = normalize(X_tfidf_reduced)   # cosine-friendly for clustering
print("TF-IDF reduced shape:", X_tfidf_reduced.shape,
      "| variance explained:", round(svd.explained_variance_ratio_.sum(), 3))

In [ ]:
# --- SBERT embeddings ---
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
X_sbert = sbert_model.encode(
    clean_df['clean_text'].tolist(),
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True
)
print("SBERT embedding shape:", X_sbert.shape)

# SBERT is already fairly low-dimensional (384-d); reduce to 100-d for a fair comparison with TF-IDF
from sklearn.decomposition import PCA
pca = PCA(n_components=100, random_state=RANDOM_STATE)
X_sbert_reduced = pca.fit_transform(X_sbert)
print("SBERT reduced shape:", X_sbert_reduced.shape,
      "| variance explained:", round(pca.explained_variance_ratio_.sum(), 3))

In [ ]:
# --- UMAP projections for visualization only (never used for clustering) ---
umap_2d_tfidf = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
X_tfidf_2d = umap_2d_tfidf.fit_transform(X_tfidf_reduced)

umap_2d_sbert = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
X_sbert_2d = umap_2d_sbert.fit_transform(X_sbert_reduced)

print("UMAP projections ready:", X_tfidf_2d.shape, X_sbert_2d.shape)

## 4. Clustering Layer
K-Means and HDBSCAN are each run on **both** representations (TF-IDF-reduced and SBERT-reduced),
giving four combinations in total. K-Means is tuned over a range of `k` using the elbow curve and
silhouette score; HDBSCAN is tuned over `min_cluster_size`.

In [ ]:
def tune_kmeans(X, k_range=range(2, 12)):
    """Sweep k, return a results dataframe and the best k by silhouette."""
    rows = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X)
        rows.append({
            'k': k,
            'inertia': km.inertia_,
            'silhouette': silhouette_score(X, labels)
        })
    results = pd.DataFrame(rows)
    best_k = int(results.loc[results['silhouette'].idxmax(), 'k'])
    return results, best_k

tfidf_kmeans_sweep, best_k_tfidf = tune_kmeans(X_tfidf_reduced)
sbert_kmeans_sweep, best_k_sbert = tune_kmeans(X_sbert_reduced)

print("Best k (TF-IDF):", best_k_tfidf)
print("Best k (SBERT) :", best_k_sbert)
tfidf_kmeans_sweep

In [ ]:
# Elbow / silhouette curves
fig = go.Figure()
fig.add_trace(go.Scatter(x=tfidf_kmeans_sweep['k'], y=tfidf_kmeans_sweep['silhouette'],
                          mode='lines+markers', name='TF-IDF'))
fig.add_trace(go.Scatter(x=sbert_kmeans_sweep['k'], y=sbert_kmeans_sweep['silhouette'],
                          mode='lines+markers', name='SBERT'))
fig.update_layout(title='K-Means: Silhouette Score vs. k', xaxis_title='k',
                   yaxis_title='Silhouette Score', template='plotly_white')
fig.show()

In [ ]:
def tune_hdbscan(X, min_cluster_sizes=(5, 10, 15, 20, 30)):
    rows = []
    for mcs in min_cluster_sizes:
        clusterer = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=max(1, mcs // 2))
        labels = clusterer.fit_predict(X)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_pct = (labels == -1).mean() * 100
        sil = silhouette_score(X, labels) if n_clusters > 1 and noise_pct < 100 else np.nan
        rows.append({'min_cluster_size': mcs, 'n_clusters': n_clusters,
                      'noise_pct': round(noise_pct, 1), 'silhouette': sil})
    results = pd.DataFrame(rows)
    valid = results.dropna(subset=['silhouette'])
    best_mcs = int(valid.loc[valid['silhouette'].idxmax(), 'min_cluster_size']) if len(valid) else min_cluster_sizes[0]
    return results, best_mcs

tfidf_hdbscan_sweep, best_mcs_tfidf = tune_hdbscan(X_tfidf_reduced)
sbert_hdbscan_sweep, best_mcs_sbert = tune_hdbscan(X_sbert_reduced)

print("Best min_cluster_size (TF-IDF):", best_mcs_tfidf)
print("Best min_cluster_size (SBERT) :", best_mcs_sbert)
sbert_hdbscan_sweep

In [ ]:
# ---- Fit final models with tuned hyperparameters ----
results = {}

for rep_name, X, X_2d, best_k, best_mcs in [
    ('TF-IDF', X_tfidf_reduced, X_tfidf_2d, best_k_tfidf, best_mcs_tfidf),
    ('SBERT',  X_sbert_reduced, X_sbert_2d, best_k_sbert, best_mcs_sbert),
]:
    km = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
    labels_km = km.fit_predict(X)

    hdb = hdbscan.HDBSCAN(min_cluster_size=best_mcs, min_samples=max(1, best_mcs // 2))
    labels_hdb = hdb.fit_predict(X)

    results[rep_name] = {
        'X': X, 'X_2d': X_2d,
        'kmeans_labels': labels_km, 'kmeans_k': best_k,
        'hdbscan_labels': labels_hdb, 'hdbscan_mcs': best_mcs
    }

print("Final models fitted for:", list(results.keys()))

## 5. Evaluation Layer
Each of the four (representation × algorithm) combinations is scored with:
- **Silhouette Score** (higher is better)
- **Davies-Bouldin Index** (lower is better)
- **Calinski-Harabasz Index** (higher is better)
- **ARI** and **NMI** against the true category labels (external validation only)

In [ ]:
def evaluate_clustering(X, labels, true_labels):
    mask = labels != -1  # exclude HDBSCAN noise points from internal metrics
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        return {'silhouette': np.nan, 'davies_bouldin': np.nan, 'calinski_harabasz': np.nan,
                'ari': np.nan, 'nmi': np.nan, 'n_clusters': 0, 'noise_pct': 100.0}
    return {
        'silhouette': silhouette_score(X[mask], labels[mask]),
        'davies_bouldin': davies_bouldin_score(X[mask], labels[mask]),
        'calinski_harabasz': calinski_harabasz_score(X[mask], labels[mask]),
        'ari': adjusted_rand_score(true_labels, labels),
        'nmi': normalized_mutual_info_score(true_labels, labels),
        'n_clusters': len(set(labels[mask])),
        'noise_pct': round((labels == -1).mean() * 100, 1)
    }

true_labels = clean_df['true_label'].values
eval_rows = []
for rep_name, r in results.items():
    for algo_name, label_key in [('K-Means', 'kmeans_labels'), ('HDBSCAN', 'hdbscan_labels')]:
        metrics = evaluate_clustering(r['X'], r[label_key], true_labels)
        metrics.update({'representation': rep_name, 'algorithm': algo_name})
        eval_rows.append(metrics)

metrics_df = pd.DataFrame(eval_rows)[
    ['representation', 'algorithm', 'n_clusters', 'noise_pct',
     'silhouette', 'davies_bouldin', 'calinski_harabasz', 'ari', 'nmi']
].round(3)
metrics_df

In [ ]:
metrics_df.to_csv('metrics.csv', index=False)
print("Saved metrics.csv")
metrics_df.sort_values('silhouette', ascending=False)

In [ ]:
def top_keywords_per_cluster(tfidf_matrix, feature_names, labels, top_n=8):
    """Mean TF-IDF of cluster members -> top N terms, used as the cluster's topic label."""
    out = {}
    feature_names = np.array(feature_names)
    for c in sorted(set(labels)):
        if c == -1:
            continue  # skip HDBSCAN noise
        idx = np.where(labels == c)[0]
        mean_scores = np.asarray(tfidf_matrix[idx].mean(axis=0)).flatten()
        top_idx = mean_scores.argsort()[::-1][:top_n]
        out[c] = list(feature_names[top_idx])
    return out

feature_names = tfidf_vectorizer.get_feature_names_out()

print("=== TF-IDF + K-Means cluster keywords ===")
for c, kws in top_keywords_per_cluster(X_tfidf, feature_names, results['TF-IDF']['kmeans_labels']).items():
    print(f"Cluster {c}: {', '.join(kws)}")

print("\n=== SBERT + K-Means cluster keywords ===")
for c, kws in top_keywords_per_cluster(X_tfidf, feature_names, results['SBERT']['kmeans_labels']).items():
    print(f"Cluster {c}: {', '.join(kws)}")

## 5b. Majority-Vote Accuracy & Confusion Matrix
Internal and external metrics (above) score cluster *geometry* and *agreement* with the true
labels, but neither is as intuitive as a plain accuracy percentage. This section maps every
predicted cluster to its majority true label, then scores the result like a classifier --
the same accuracy figure and confusion matrix reported in the project e-report and slides.

HDBSCAN noise points (`-1`) are left unmapped and always counted as incorrect, since "noise"
is not a category prediction.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import Counter

def cluster_accuracy_and_confusion(pred_labels, true_labels, label_names):
    """Map each predicted cluster to its majority true label, then score like a classifier."""
    pred_labels = np.asarray(pred_labels)
    true_labels = np.asarray(true_labels)
    mapping = {}
    for cluster_id in set(pred_labels):
        if cluster_id == -1:
            continue  # HDBSCAN noise points excluded from the mapping step
        mask = (pred_labels == cluster_id)
        majority_label = Counter(true_labels[mask]).most_common(1)[0][0]
        mapping[cluster_id] = majority_label
    # noise / unmapped points get a label that can never match -> always scored wrong
    mapped_preds = [mapping.get(c, -1) for c in pred_labels]
    acc = accuracy_score(true_labels, mapped_preds)
    cm = confusion_matrix(true_labels, mapped_preds, labels=range(len(label_names)))
    return acc, cm, mapping

print("cluster_accuracy_and_confusion() ready.")

In [ ]:
# Majority-vote accuracy for all four representation x algorithm combinations
accuracy_rows = []
confusions = {}
for rep_name, r in results.items():
    for algo_name, label_key in [('K-Means', 'kmeans_labels'), ('HDBSCAN', 'hdbscan_labels')]:
        acc, cm, _ = cluster_accuracy_and_confusion(r[label_key], true_labels, label_names)
        accuracy_rows.append({'representation': rep_name, 'algorithm': algo_name, 'accuracy': round(acc, 3)})
        confusions[(rep_name, algo_name)] = cm

accuracy_df = pd.DataFrame(accuracy_rows)
metrics_df = metrics_df.merge(accuracy_df, on=['representation', 'algorithm'], how='left')
metrics_df.to_csv('metrics.csv', index=False)  # re-save with the accuracy column included
metrics_df.sort_values('accuracy', ascending=False)

## 6. Visualization Layer\nInteractive UMAP scatter plots, colored by cluster, for each representation × algorithm combination.

In [ ]:
def plot_clusters(X_2d, labels, title, doc_texts, true_labels=None, label_names=None):
    hover_text = [t[:120] + ('...' if len(t) > 120 else '') for t in doc_texts]
    label_strs = labels.astype(str)
    label_strs = np.where(labels == -1, 'noise', label_strs)

    fig = px.scatter(
        x=X_2d[:, 0], y=X_2d[:, 1],
        color=label_strs,
        hover_name=hover_text,
        title=title,
        labels={'x': 'UMAP-1', 'y': 'UMAP-2', 'color': 'Cluster'},
        opacity=0.75, width=850, height=550
    )
    fig.update_layout(template='plotly_white', legend_title_text='Cluster')
    return fig

doc_texts = raw_df['text'].tolist()

plot_clusters(results['TF-IDF']['X_2d'], results['TF-IDF']['kmeans_labels'],
              'TF-IDF + K-Means', doc_texts).show()
plot_clusters(results['TF-IDF']['X_2d'], results['TF-IDF']['hdbscan_labels'],
              'TF-IDF + HDBSCAN', doc_texts).show()
plot_clusters(results['SBERT']['X_2d'], results['SBERT']['kmeans_labels'],
              'SBERT + K-Means', doc_texts).show()
plot_clusters(results['SBERT']['X_2d'], results['SBERT']['hdbscan_labels'],
              'SBERT + HDBSCAN', doc_texts).show()

In [ ]:
# Confusion matrix heatmap for the strongest combination (SBERT + K-Means)
import seaborn as sns
import matplotlib.pyplot as plt

best_repr, best_algo = 'SBERT', 'K-Means'  # established above and in metrics_df as the top result
best_acc, best_cm, _ = cluster_accuracy_and_confusion(
    results[best_repr]['kmeans_labels'], true_labels, label_names)

plt.figure(figsize=(7, 6))
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, cbar=True)
plt.xlabel('Predicted (cluster \u2192 majority label)')
plt.ylabel('True category')
plt.title(f'Confusion Matrix \u2014 Accuracy: {best_acc:.3f}  ({best_repr} + {best_algo})')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print(f"Majority-vote accuracy ({best_repr} + {best_algo}): {best_acc:.3f}")

In [ ]:
# Side-by-side bar chart comparing all four combinations across metrics
compare_df = metrics_df.copy()
compare_df['combo'] = compare_df['representation'] + ' + ' + compare_df['algorithm']

fig = go.Figure()
for metric in ['silhouette', 'ari', 'nmi']:
    fig.add_trace(go.Bar(name=metric, x=compare_df['combo'], y=compare_df[metric]))
fig.update_layout(barmode='group', title='Metric comparison across representation x algorithm',
                   template='plotly_white', yaxis_title='Score')
fig.show()

## 7. Summary

Run the cell below to print the best-performing combination and a short written summary you can
paste directly into your report / demo.

In [ ]:
# Pick the best combination among results with usable coverage (noise < 50%),
# ranked by ARI -- the metric actually checked against the true category labels.
# (Silhouette alone is unreliable here: HDBSCAN's internal metrics are computed only
#  on the points it keeps, so a combination that marks most documents as noise can
#  post an artificially high silhouette on the easy minority it didn't discard.)
candidates = metrics_df[metrics_df['noise_pct'] < 50]
best_row = candidates.sort_values('ari', ascending=False).iloc[0]

print("BEST COMBINATION")
print("-----------------")
print(f"Representation     : {best_row['representation']}")
print(f"Algorithm          : {best_row['algorithm']}")
print(f"Clusters found     : {int(best_row['n_clusters'])}")
print(f"Noise %            : {best_row['noise_pct']}")
print(f"Silhouette         : {best_row['silhouette']:.3f}")
print(f"Davies-Bouldin     : {best_row['davies_bouldin']:.3f}")
print(f"Calinski-Harabasz  : {best_row['calinski_harabasz']:.1f}")
print(f"ARI vs ground truth: {best_row['ari']:.3f}")
print(f"NMI vs ground truth: {best_row['nmi']:.3f}")
print()
print("Selection rule: highest ARI among combinations with < 50% noise --")
print("ARI/NMI are checked against the true labels, so they're a fairer basis")
print("for picking an overall winner than silhouette alone (see note above).")


---
### Notes for your report
- Metrics are saved to `metrics.csv` in this notebook's working directory.
- To switch datasets, edit the **Data Ingestion** cell (Option A/B) — every downstream layer is unchanged.
- `CATEGORIES` in Layer 1 controls dataset size/runtime; set to `None` to use all 20 categories (slower).
- ARI/NMI are **external** metrics (need true labels) — set `true_label = -1` for all rows if you don't have ground truth, and drop those two columns from your reported results.